In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("codespaces-spark") \
    .config("spark.ui.port", "4050") \
    .config("spark.ui.enabled", "true") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.local.dir", "/tmp") \
    .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse") \
    .getOrCreate()

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/07 08:13:44 WARN Utils: Your hostname, codespaces-b285a2, resolves to a loopback address: 127.0.0.1; using 10.0.2.238 instead (on interface eth0)
26/03/07 08:13:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 08:13:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/07 08:13:52 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


Spark version: 4.1.1


In [3]:
!wget -O /tmp/fhvhv_tripdata_2021-01.csv.gz https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-03-07 08:14:13--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 20.207.73.82
Connecting to github.com (github.com)|20.207.73.82|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-07T09%3A08%3A26Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-07T08%3A08%3A24Z&ske=2026-03-07T09%3A08%3A26Z&sks=b&skv=2018-11-09&sig=fdSccvEAoRwyFz8UIGf6xq1yYWVQIAYZPJsQIHMIIps%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3Mjg3NDg1MywibmJmIjoxNzcyODcxMjUzLCJwYXRoIjoi

In [4]:
!ls -lh /tmp/fhvhv_tripdata_2021-01.csv.gz

-rw-r--rw- 1 codespace codespace 124M Jul 14  2022 /tmp/fhvhv_tripdata_2021-01.csv.gz


In [5]:
!gzip -dc /tmp/fhvhv_tripdata_2021-01.csv.gz > /tmp/fhvhv_tripdata_2021-01.csv

In [6]:
!wc -l /tmp/fhvhv_tripdata_2021-01.csv

11908469 /tmp/fhvhv_tripdata_2021-01.csv


In [7]:
df = spark.read \
    .option("header", "true") \
    .csv("/tmp/fhvhv_tripdata_2021-01.csv")

In [8]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [9]:
!head -n 1001 /tmp/fhvhv_tripdata_2021-01.csv > /tmp/head.csv

In [10]:
import pandas as pd

In [11]:
df_pandas = pd.read_csv("/tmp/head.csv")

In [12]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [13]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [14]:
from pyspark.sql import types

In [15]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [16]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('/tmp/fhvhv_tripdata_2021-01.csv')

In [17]:
df = df.repartition(24)

In [18]:
df.write.parquet('/tmp/fhvhv/2021/01/', mode='overwrite')

In [19]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [20]:
!ls -lh /tmp/fhvhv/2021/01/

total 215M
-rw-r--r-- 1 codespace codespace    0 Mar  7 08:16 _SUCCESS
-rw-r--r-- 1 codespace codespace 9.0M Mar  7 08:16 part-00000-d719c0e1-61c1-44c9-9bfb-cf63bfffcf26-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 9.0M Mar  7 08:16 part-00001-d719c0e1-61c1-44c9-9bfb-cf63bfffcf26-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 9.0M Mar  7 08:16 part-00002-d719c0e1-61c1-44c9-9bfb-cf63bfffcf26-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 9.0M Mar  7 08:16 part-00003-d719c0e1-61c1-44c9-9bfb-cf63bfffcf26-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 9.0M Mar  7 08:16 part-00004-d719c0e1-61c1-44c9-9bfb-cf63bfffcf26-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 9.0M Mar  7 08:16 part-00005-d719c0e1-61c1-44c9-9bfb-cf63bfffcf26-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 9.0M Mar  7 08:16 part-00006-d719c0e1-61c1-44c9-9bfb-cf63bfffcf26-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 9.0M Mar  7 08:16 part-00007-d719c0e1-61c1-44c9-9bfb-cf63bfffcf

In [36]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003') \
.head(5)

[Row(pickup_datetime=datetime.datetime(2021, 1, 3, 17, 33, 56), dropoff_datetime=datetime.datetime(2021, 1, 3, 17, 41, 52), PULocationID=90, DOLocationID=231),
 Row(pickup_datetime=datetime.datetime(2021, 1, 1, 1, 50, 24), dropoff_datetime=datetime.datetime(2021, 1, 1, 2, 20, 31), PULocationID=260, DOLocationID=165),
 Row(pickup_datetime=datetime.datetime(2021, 1, 1, 12, 1, 58), dropoff_datetime=datetime.datetime(2021, 1, 1, 12, 21, 33), PULocationID=42, DOLocationID=138),
 Row(pickup_datetime=datetime.datetime(2021, 1, 4, 16, 19, 51), dropoff_datetime=datetime.datetime(2021, 1, 4, 16, 51, 42), PULocationID=138, DOLocationID=106),
 Row(pickup_datetime=datetime.datetime(2021, 1, 2, 16, 53, 49), dropoff_datetime=datetime.datetime(2021, 1, 2, 17, 20, 56), PULocationID=43, DOLocationID=100)]

In [25]:
from pyspark.sql import functions as F

In [27]:
df.show()

[Stage 7:=================================================>         (5 + 1) / 6]

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02765|2021-01-04 10:59:08|2021-01-04 11:06:48|         255|         112|   NULL|
|           HV0003|              B02878|2021-01-06 09:37:31|2021-01-06 09:44:47|         226|         157|   NULL|
|           HV0005|              B02510|2021-01-04 13:10:40|2021-01-04 13:34:40|         141|         138|   NULL|
|           HV0003|              B02882|2021-01-03 15:11:11|2021-01-03 15:30:16|         243|          24|   NULL|
|           HV0003|              B02882|2021-01-07 05:19:55|2021-01-07 05:48:38|          76|         117|   NULL|
|           HV0003|              B02887|2021-01-03 12:40:52|2021-01-03 12:54:01|

In [28]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [29]:
crazy_stuff('B02884')

's/b44'

In [30]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [31]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

[Stage 12:>                                                         (0 + 1) / 1]

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  a/a7a| 2021-01-05|  2021-01-05|         218|         130|
|  e/9ce| 2021-01-04|  2021-01-04|         145|         146|
|  e/b33| 2021-01-01|  2021-01-01|         179|         179|
|  e/b3b| 2021-01-01|  2021-01-01|         225|         177|
|  e/9ce| 2021-01-03|  2021-01-03|         181|          61|
|  e/9ce| 2021-01-06|  2021-01-06|          74|          41|
|  s/b13| 2021-01-01|  2021-01-01|          59|         212|
|  s/acd| 2021-01-03|  2021-01-03|          49|         217|
|  e/a39| 2021-01-06|  2021-01-06|          76|         188|
|  e/acc| 2021-01-06|  2021-01-06|         225|          80|
|  a/b37| 2021-01-02|  2021-01-02|           4|         181|
|  e/acc| 2021-01-07|  2021-01-07|          42|         166|
|  e/b3b| 2021-01-03|  2021-01-03|         244|         143|
|  s/b36| 2021-01-01|  2

In [ ]:
spark.stop()

In [64]:
#Cleanup cell AFTER jobs
import shutil, os

def clear_tmp():
    tmp_path = "/tmp/spark-warehouse"
    if os.path.exists(tmp_path):
        shutil.rmtree(tmp_path)
    print("Cleared /tmp Spark warehouse")

clear_tmp()

Cleared /tmp Spark warehouse


In [65]:
!ls -lh /tmp

total 842M
drwxr-xrw-+  2 codespace codespace 4.0K Mar  4 09:06 artifacts-30fd8fbf-f5d2-465c-95ed-f935cff3a4b7
drwxr-xrw-+ 66 codespace codespace 4.0K Mar  4 09:52 blockmgr-3208d09e-96ad-4c9a-af80-40608eab457d
-rw-r--rw-   1 root      root       12K Mar  4 07:12 dockerd.log
drwxr-xr-x+  3 codespace codespace 4.0K Mar  4 09:52 fhvhv
-rw-r--rw-   1 codespace codespace 718M Mar  4 09:44 fhvhv_tripdata_2021-01.csv
-rw-r--rw-   1 codespace codespace 124M Jul 14  2022 fhvhv_tripdata_2021-01.csv.gz
-rw-r--rw-   1 codespace codespace  62K Mar  4 09:45 head.csv
drwxr-xr--+  2 codespace codespace 4.0K Mar  4 09:06 hsperfdata_codespace
-rw-r--rw-   1 codespace codespace 152K Mar  4 09:12 liblz4-java-8498321997344148136.so
-rw-r--rw-   1 codespace codespace    0 Mar  4 09:12 liblz4-java-8498321997344148136.so.lck
drwx------+  2 codespace codespace 4.0K Mar  4 08:28 mcp-5CWuZP
drwx------+  2 codespace codespace 4.0K Mar  4 07:34 mcp-GqWa9J
drwx------+  2 codespace codespace 4.0K Mar  4 07:27 mcp-Kl